# Experiment: Feature, Model, Target & Signal Lab

Use this notebook as a repo-native sandbox for inspecting the current `features`, `models`, `targets`, and `signals` surface, then running small experiments without leaving the notebook.


## Objective

This notebook is designed as the broader companion to [`feature_indicator_visual_lab.ipynb`](./feature_indicator_visual_lab.ipynb):

- it imports the current project callables that matter for experimentation,
- shows the experiment-facing registries and their signatures/config levers,
- explains what each feature/model/target/signal does,
- gives you one playground cell per stage so you can tweak parameters incrementally.

The important mapping to the YAML experiment system is:

- `features[].step` -> `FEATURE_REGISTRY`
- `model.kind` or `model_stages[].kind` -> `MODEL_REGISTRY`
- `model.target.kind` or `model_stages[].target.kind` -> target builders
- `signals.kind` -> `SIGNAL_REGISTRY`


In [ ]:
from __future__ import annotations

from inspect import getdoc, signature
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / '.git').exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import src.features as feature_pkg
import src.models as model_pkg
import src.signals as signal_pkg
import src.targets as target_pkg
from src.experiments.models import (
    infer_feature_columns,
    train_dqn_agent,
    train_dqn_portfolio_agent,
    train_event_transformer_encoder,
    train_garch_forecaster,
    train_lightgbm_classifier,
    train_logistic_regression_classifier,
    train_lstm_forecaster,
    train_patchtst_forecaster,
    train_ppo_agent,
    train_ppo_portfolio_agent,
    train_sarimax_forecaster,
    train_tft_forecaster,
    train_xgboost_classifier,
)
from src.experiments.registry import (
    FEATURE_REGISTRY,
    MODEL_REGISTRY,
    PORTFOLIO_MODEL_REGISTRY,
    SIGNAL_REGISTRY,
    SINGLE_ASSET_MODEL_REGISTRY,
)
from src.features import (
    add_adx_features,
    add_atr_features,
    add_bollinger_features,
    add_close_returns,
    add_feature_transforms,
    add_lagged_features,
    add_macro_context_features,
    add_macd_features,
    add_mfi_features,
    add_momentum_features,
    add_oscillator_features,
    add_ppo_features,
    add_price_momentum_features,
    add_regime_context_features,
    add_return_momentum_features,
    add_roc_features,
    add_rsi_features,
    add_session_context_features,
    add_shock_context_features,
    add_stochastic_features,
    add_support_resistance_features,
    add_support_resistance_v2_features,
    add_trend_features,
    add_trend_regime_features,
    add_vol_normalized_momentum_features,
    add_volatility_features,
    add_volume_features,
    compute_ema,
    compute_ewma_vol,
    compute_returns,
    compute_rolling_clip_transform,
    compute_rolling_vol,
    compute_sma,
)
from src.models.runtime import probe_lightgbm_runtime, probe_xgboost_runtime
from src.signals import (
    buy_and_hold_signal,
    compute_forecast_threshold_signal,
    compute_forecast_vol_adjusted_signal,
    compute_momentum_signal,
    compute_probability_vol_adjusted_signal,
    compute_rsi_signal,
    compute_stochastic_signal,
    compute_trend_state_signal,
    compute_volatility_regime_signal,
    conviction_sizing_signal,
    forecast_threshold_signal,
    forecast_vol_adjusted_signal,
    momentum_strategy,
    probability_vol_adjusted_signal,
    probabilistic_signal,
    regime_filtered_signal,
    rsi_strategy,
    stochastic_strategy,
    trend_state_long_only_signal,
    trend_state_signal,
    vol_targeted_signal,
    volatility_regime_strategy,
)
from src.src_data import load_ohlcv_csv
from src.targets import (
    assign_quantile_labels,
    build_classifier_target,
    build_forward_return_target,
    build_triple_barrier_target,
)
from src.utils.config import load_experiment_config

try:
    import optuna as optuna_pkg
    from src.experiments.optuna_search import (
        ConstraintPenalty,
        ObjectiveSpec,
        PruningSpec,
        SearchDimension,
        load_search_space_yaml,
        optimize_experiment,
    )
    OPTUNA_NOTEBOOK_AVAILABLE = True
    OPTUNA_NOTEBOOK_IMPORT_ERROR = None
except Exception as exc:
    optuna_pkg = None
    ConstraintPenalty = None
    ObjectiveSpec = None
    PruningSpec = None
    SearchDimension = None
    load_search_space_yaml = None
    optimize_experiment = None
    OPTUNA_NOTEBOOK_AVAILABLE = False
    OPTUNA_NOTEBOOK_IMPORT_ERROR = f"{type(exc).__name__}: {exc}"

pd.options.display.max_rows = 200
pd.options.display.max_colwidth = 140
pd.options.display.float_format = '{:,.6f}'.format


In [ ]:
if "compute_returns" not in globals():
    import json
    from pathlib import Path

    _bootstrap_repo_root = Path.cwd()
    while _bootstrap_repo_root != _bootstrap_repo_root.parent and not (_bootstrap_repo_root / '.git').exists():
        _bootstrap_repo_root = _bootstrap_repo_root.parent

    _bootstrap_nb_path = _bootstrap_repo_root / 'notebooks' / 'feature_model_target_signal_lab.ipynb'
    _bootstrap_nb = json.loads(_bootstrap_nb_path.read_text())
    _setup_source = _bootstrap_nb['cells'][2]['source']
    exec(compile(_setup_source, f'{_bootstrap_nb_path.name}:cell_2', 'exec'), globals(), globals())

FEATURE_IMPORTS = {
    fn.__name__: fn
    for fn in [
        compute_returns,
        add_close_returns,
        add_volatility_features,
        compute_rolling_vol,
        compute_ewma_vol,
        add_trend_features,
        add_trend_regime_features,
        compute_sma,
        compute_ema,
        add_lagged_features,
        add_bollinger_features,
        add_macd_features,
        add_ppo_features,
        add_roc_features,
        add_atr_features,
        add_adx_features,
        add_volume_features,
        add_mfi_features,
        add_rsi_features,
        add_stochastic_features,
        add_price_momentum_features,
        add_return_momentum_features,
        add_vol_normalized_momentum_features,
        add_session_context_features,
        add_regime_context_features,
        add_shock_context_features,
        add_macro_context_features,
        add_support_resistance_features,
        add_support_resistance_v2_features,
        add_feature_transforms,
        compute_rolling_clip_transform,
        add_momentum_features,
        add_oscillator_features,
    ]
}

MODEL_IMPORTS = {
    fn.__name__: fn
    for fn in [
        infer_feature_columns,
        train_lightgbm_classifier,
        train_logistic_regression_classifier,
        train_xgboost_classifier,
        train_event_transformer_encoder,
        train_sarimax_forecaster,
        train_garch_forecaster,
        train_lstm_forecaster,
        train_patchtst_forecaster,
        train_tft_forecaster,
        train_ppo_agent,
        train_dqn_agent,
        train_ppo_portfolio_agent,
        train_dqn_portfolio_agent,
    ]
}

TARGET_IMPORTS = {
    fn.__name__: fn
    for fn in [
        assign_quantile_labels,
        build_classifier_target,
        build_forward_return_target,
        build_triple_barrier_target,
    ]
}

SIGNAL_IMPORTS = {
    fn.__name__: fn
    for fn in [
        buy_and_hold_signal,
        conviction_sizing_signal,
        compute_forecast_threshold_signal,
        compute_forecast_vol_adjusted_signal,
        compute_momentum_signal,
        compute_probability_vol_adjusted_signal,
        compute_rsi_signal,
        compute_stochastic_signal,
        compute_trend_state_signal,
        compute_volatility_regime_signal,
        forecast_threshold_signal,
        forecast_vol_adjusted_signal,
        momentum_strategy,
        probability_vol_adjusted_signal,
        probabilistic_signal,
        regime_filtered_signal,
        rsi_strategy,
        stochastic_strategy,
        trend_state_long_only_signal,
        trend_state_signal,
        vol_targeted_signal,
        volatility_regime_strategy,
    ]
}

OPTUNA_IMPORTS = {}
if OPTUNA_NOTEBOOK_AVAILABLE:
    for fn in [
        SearchDimension,
        ObjectiveSpec,
        ConstraintPenalty,
        PruningSpec,
        load_search_space_yaml,
        optimize_experiment,
    ]:
        OPTUNA_IMPORTS[fn.__name__] = fn

TARGET_BUILDERS = {
    'forward_return': build_forward_return_target,
    'triple_barrier': build_triple_barrier_target,
    'classifier_router': build_classifier_target,
}

FEATURE_NOTES = {
    'returns': {'group': 'momentum', 'what': 'Simple or log returns from a selected price series.', 'tuning': 'Switch log/simple and name the downstream returns column you want to reuse.'},
    'volatility': {'group': 'volatility', 'what': 'Rolling and EWMA volatility from an existing returns column.', 'tuning': 'Tune lookback windows and annualization only if the downstream consumer expects it.'},
    'trend': {'group': 'trend', 'what': 'SMA and EMA overlays plus price-over-average ratios.', 'tuning': 'Tune fast/slow windows; ratios are usually more model-friendly than raw moving averages.'},
    'trend_regime': {'group': 'trend', 'what': 'Short-vs-long trend state context that compresses trend structure into a regime label.', 'tuning': 'Tune short/long windows conservatively to avoid regime overfit.'},
    'lags': {'group': 'transforms', 'what': 'Lagged copies of existing columns for strictly causal comparisons.', 'tuning': 'Choose only the lags you are willing to pay for in dimensionality.'},
    'bollinger': {'group': 'trend/volatility', 'what': 'Bollinger moving average, bands, width, and percent-b features.', 'tuning': 'Tune window and standard deviation multiplier together.'},
    'macd': {'group': 'momentum', 'what': 'MACD line, signal line, and histogram from EMA spreads.', 'tuning': 'Fast/slow/signal spans control responsiveness vs noise.'},
    'ppo': {'group': 'momentum', 'what': 'Percentage Price Oscillator, useful when scale invariance matters.', 'tuning': 'Tune fast/slow spans similarly to MACD, especially across assets with different price levels.'},
    'roc': {'group': 'momentum', 'what': 'Rate-of-change over one or more horizons.', 'tuning': 'Good search space candidate for lookback windows.'},
    'atr': {'group': 'volatility', 'what': 'Average True Range and optional ATR-over-price normalization.', 'tuning': 'Tune window and whether normalized ATR is more useful than absolute ATR.'},
    'adx': {'group': 'trend', 'what': 'Trend-strength feature independent of direction.', 'tuning': 'Usually paired with trend or breakout features; avoid too many overlapping windows.'},
    'volume_features': {'group': 'volume', 'what': 'Volume z-score and volume-vs-range context.', 'tuning': 'Tune z-score windows and any ATR-normalized ratio thresholds.'},
    'mfi': {'group': 'volume', 'what': 'Money Flow Index using price and volume together.', 'tuning': 'Tune window only; use mainly as confirmation or meta-model input.'},
    'rsi': {'group': 'momentum', 'what': 'Relative Strength Index over one or more windows.', 'tuning': 'Tune windows and Wilder vs simple method based on indicator smoothness needs.'},
    'stochastic': {'group': 'momentum', 'what': 'Stochastic oscillator from close relative to recent high-low range.', 'tuning': 'Tune %K, %D, and smoothing windows.'},
    'price_momentum': {'group': 'momentum', 'what': 'Price-level momentum over multiple horizons.', 'tuning': 'Tune horizons and use alongside trend filters, not as a standalone kitchen sink.'},
    'return_momentum': {'group': 'momentum', 'what': 'Momentum computed directly from returns.', 'tuning': 'Useful when you want additive return dynamics instead of price ratios.'},
    'vol_normalized_momentum': {'group': 'momentum/volatility', 'what': 'Momentum scaled by recent volatility for cross-regime comparability.', 'tuning': 'Tune momentum and volatility windows jointly.'},
    'session_context': {'group': 'context', 'what': 'Hour, weekday, and session membership context features.', 'tuning': 'Useful for intraday assets; keep timezone assumptions explicit.'},
    'regime_context': {'group': 'context', 'what': 'Higher-level volatility/trend regime labels.', 'tuning': 'Tune thresholds only with strong OOS validation because regime cuts overfit easily.'},
    'shock_context': {'group': 'context', 'what': 'Shock and reversion context from returns, ATR, and mean-reversion distance.', 'tuning': 'Tune shock thresholds and holding assumptions with care.'},
    'support_resistance': {'group': 'structure', 'what': 'Rolling support/resistance levels and distance-to-level features.', 'tuning': 'Tune rolling windows and distance normalization carefully to keep causality.'},
    'support_resistance_v2': {'group': 'structure', 'what': 'Pivot-confirmed support/resistance with touch and breakout context.', 'tuning': 'Tune pivot confirmation and tolerance windows; this is the PIT-safer structure path.'},
    'macro_context': {'group': 'context', 'what': 'Macro or exogenous context aligned onto market timestamps.', 'tuning': 'Only use properly lagged macro joins and explicit release timing.'},
    'feature_transforms': {'group': 'transforms', 'what': 'Rolling z-score, clipping, ratios, and other post-processing transforms.', 'tuning': 'Use for normalization and robustification, not as a substitute for better raw features.'},
}

MODEL_NOTES = {
    'lightgbm_clf': {'family': 'classification', 'what': 'Tree-based nonlinear directional classifier with native probability output.', 'config_keys': 'params, feature_cols, target, split, runtime, preprocessing, pred_prob_col, overlay', 'when': 'Strong default when interactions matter and tabular feature quality is already good.'},
    'logistic_regression_clf': {'family': 'classification', 'what': 'Linear baseline classifier with stable behaviour and fast walk-forward fits.', 'config_keys': 'params, feature_cols, target, split, runtime, preprocessing, pred_prob_col, overlay', 'when': 'Best first baseline for checking whether the feature set has directional signal at all.'},
    'xgboost_clf': {'family': 'classification', 'what': 'Boosted-tree classifier for more flexible nonlinear event modeling.', 'config_keys': 'params, feature_cols, target, split, runtime, preprocessing, pred_prob_col, overlay', 'when': 'Useful after the linear baseline is exhausted and you still want tabular flexibility.'},
    'event_transformer_encoder': {'family': 'classification', 'what': 'Sequence/event encoder for classification on event-like temporal feature windows.', 'config_keys': 'params, feature_cols, event_embedding_cols, target, split, runtime, pred_prob_col', 'when': 'Use when local event sequences matter more than static tabular snapshots.'},
    'sarimax_forecaster': {'family': 'forecasting', 'what': 'Classical SARIMAX forecaster with optional exogenous features.', 'config_keys': 'params, target, split, pred_ret_col, pred_prob_col, feature_cols, use_features', 'when': 'Good low-variance forecasting baseline when seasonality or autoregression dominates.'},
    'garch_forecaster': {'family': 'forecasting', 'what': 'Volatility-oriented forecaster for conditional variance and related overlays.', 'config_keys': 'params, target, split, pred_ret_col, pred_prob_col, returns_col', 'when': 'Useful mainly as a volatility/risk overlay rather than the primary alpha model.'},
    'lstm_forecaster': {'family': 'forecasting', 'what': 'Recurrent neural forecaster for sequence-dependent forward returns.', 'config_keys': 'params, target, split, pred_ret_col, pred_prob_col, feature_cols, runtime, use_features', 'when': 'Use only when you already have evidence that temporal state carries signal beyond tabular lags.'},
    'patchtst_forecaster': {'family': 'forecasting', 'what': 'Patch-based transformer forecaster for long-horizon sequence structure.', 'config_keys': 'params, target, split, pred_ret_col, pred_prob_col, feature_cols, runtime, use_features', 'when': 'More appropriate for richer sequence forecasting setups than for quick baseline work.'},
    'tft_forecaster': {'family': 'forecasting', 'what': 'Temporal Fusion Transformer forecaster with richer temporal covariate handling.', 'config_keys': 'params, target, split, pred_ret_col, pred_prob_col, feature_cols, runtime, use_features', 'when': 'Use if you explicitly need multi-horizon neural forecasting with heterogeneous temporal covariates.'},
    'ppo_agent': {'family': 'reinforcement_learning', 'what': 'Single-asset PPO policy optimization agent.', 'config_keys': 'params, split, runtime, reward, execution, environment', 'when': 'Use only after strong supervised baselines; RL search spaces are much easier to overfit.'},
    'dqn_agent': {'family': 'reinforcement_learning', 'what': 'Single-asset discrete-action DQN agent.', 'config_keys': 'params, split, runtime, reward, execution, environment', 'when': 'Useful only for clearly discretized trading actions and tightly scoped research questions.'},
    'ppo_portfolio_agent': {'family': 'portfolio_rl', 'what': 'Multi-asset PPO portfolio allocator.', 'config_keys': 'params, split, runtime, reward, execution, portfolio, environment', 'when': 'Research-only path when you need weight allocation instead of single-asset direction.'},
    'dqn_portfolio_agent': {'family': 'portfolio_rl', 'what': 'Multi-asset discrete-action portfolio RL agent.', 'config_keys': 'params, split, runtime, reward, execution, portfolio, environment', 'when': 'Use sparingly; discrete portfolio action spaces become brittle quickly.'},
}

TARGET_NOTES = {
    'forward_return': {'what': 'Canonical forward-return regression target plus optional binary label.', 'config_keys': 'price_col, returns_col, returns_type, horizon, fwd_col, label_col, threshold, quantiles', 'when': 'Use for both direct regression and simple sign classification.'},
    'triple_barrier': {'what': 'Event-based label from upper/lower/vertical barriers with event return and hit-step outputs.', 'config_keys': 'price_col, open_col, high_col, low_col, returns_col, volatility_col, label_col, event_ret_col, fwd_col, max_holding, upper_mult, lower_mult, neutral_label, tie_break, vol_window, min_vol, side_col, candidate_col, candidate_mode, candidate_out_col', 'when': 'Use for direction models and meta-labeling where event timing matters.'},
    'classifier_router': {'what': 'Router used by classifiers to dispatch to forward_return or triple_barrier based on target.kind.', 'config_keys': 'kind plus the config keys of the selected concrete target', 'when': 'Use when you want the same training code to switch target families without changing model wiring.'},
}

SIGNAL_NOTES = {
    'trend_state': {'what': 'Turns a discrete trend-state column into a directional signal.', 'tuning': 'Choose mode and state column semantics carefully.'},
    'probability_threshold': {'what': 'Thresholds classifier probabilities into long/short/flat positions with optional hysteresis.', 'tuning': 'Upper/lower and exit thresholds are the main control surface.'},
    'probability_conviction': {'what': 'Maps probability distance from a center point into conviction-weighted sizing.', 'tuning': 'Tune probability center, scaling, and clipping.'},
    'probability_vol_adjusted': {'what': 'Scales probability-based conviction by predicted volatility and optional activation filters.', 'tuning': 'Tune vol_target, clip, min_signal_abs, and any activation filters.'},
    'forecast_threshold': {'what': 'Thresholds predicted returns into direction.', 'tuning': 'Tune return thresholds and mode.'},
    'forecast_vol_adjusted': {'what': 'Converts return forecasts into risk-adjusted sizing using a volatility column.', 'tuning': 'Tune clip, vol_floor, and choice of volatility forecast.'},
    'rsi': {'what': 'Strategy wrapper that maps RSI regimes to trade signals.', 'tuning': 'Tune RSI windows and overbought/oversold thresholds together.'},
    'momentum': {'what': 'Strategy wrapper for momentum-based signal generation.', 'tuning': 'Tune lookback horizon and entry threshold conservatively.'},
    'stochastic': {'what': 'Strategy wrapper for stochastic oscillator signals.', 'tuning': 'Tune %K/%D thresholds and hold mode.'},
    'volatility_regime': {'what': 'Trades only when the selected volatility regime condition is active.', 'tuning': 'Tune regime thresholds and whether the filter should block or scale exposure.'},
}


def first_doc_line(fn):
    doc = getdoc(fn) or ''
    return doc.splitlines()[0].strip() if doc else ''


def parameter_overview(fn, *, skip=('df',)):
    parts = []
    for name, param in signature(fn).parameters.items():
        if name in skip:
            continue
        if param.default is param.empty:
            default = 'required'
        else:
            default = repr(param.default)
        parts.append(f"{name}={default}")
    return ', '.join(parts)


def existing_columns(frame, columns):
    return [column for column in columns if column in frame.columns]


def apply_feature(frame, feature_fn, **params):
    before_cols = list(frame.columns)
    out = feature_fn(frame.copy(), **params)
    new_cols = [column for column in out.columns if column not in before_cols]
    return out, new_cols


def show_df(title, frame, *, rows=20):
    display(Markdown(f"### {title}"))
    if isinstance(frame, pd.Series):
        display(frame.to_frame().head(rows))
    else:
        display(frame.head(rows))


def build_feature_catalog():
    rows = []
    for kind, fn in FEATURE_REGISTRY.items():
        note = FEATURE_NOTES[kind]
        rows.append({
            'kind': kind,
            'config_path': 'features[].step',
            'group': note['group'],
            'callable': fn.__name__,
            'signature': str(signature(fn)),
            'parameters': parameter_overview(fn),
            'what_it_does': note['what'],
            'tuning_focus': note['tuning'],
            'doc': first_doc_line(fn),
        })
    return pd.DataFrame(rows).sort_values(['group', 'kind']).reset_index(drop=True)


def build_model_catalog():
    rows = []
    for kind, fn in MODEL_REGISTRY.items():
        note = MODEL_NOTES[kind]
        rows.append({
            'kind': kind,
            'config_path': 'model.kind or model_stages[].kind',
            'family': note['family'],
            'callable': fn.__name__,
            'signature': str(signature(fn)),
            'model_cfg_keys': note['config_keys'],
            'what_it_does': note['what'],
            'when_to_use': note['when'],
            'doc': first_doc_line(fn),
        })
    return pd.DataFrame(rows).sort_values(['family', 'kind']).reset_index(drop=True)


def build_target_catalog():
    rows = []
    for kind, fn in TARGET_BUILDERS.items():
        note = TARGET_NOTES[kind]
        rows.append({
            'kind': kind,
            'config_path': 'model.target.kind / model_stages[].target.kind',
            'callable': fn.__name__,
            'signature': str(signature(fn)),
            'target_cfg_keys': note['config_keys'],
            'what_it_does': note['what'],
            'when_to_use': note['when'],
            'doc': first_doc_line(fn),
        })
    return pd.DataFrame(rows).reset_index(drop=True)


def build_signal_catalog():
    rows = []
    for kind, fn in SIGNAL_REGISTRY.items():
        note = SIGNAL_NOTES[kind]
        rows.append({
            'kind': kind,
            'config_path': 'signals.kind',
            'callable': fn.__name__,
            'signature': str(signature(fn)),
            'parameters': parameter_overview(fn),
            'what_it_does': note['what'],
            'tuning_focus': note['tuning'],
            'doc': first_doc_line(fn),
        })
    return pd.DataFrame(rows).sort_values('kind').reset_index(drop=True)


def probe_tree_runtimes():
    rows = []
    for name, probe in [('lightgbm', probe_lightgbm_runtime), ('xgboost', probe_xgboost_runtime)]:
        try:
            ok, detail = probe()
        except Exception as exc:
            ok, detail = False, f'{type(exc).__name__}: {exc}'
        rows.append({'runtime': name, 'available': bool(ok), 'detail': detail})
    return pd.DataFrame(rows)


feature_catalog = build_feature_catalog()
model_catalog = build_model_catalog()
target_catalog = build_target_catalog()
signal_catalog = build_signal_catalog()

tree_runtime_status = probe_tree_runtimes()
optuna_runtime_status = pd.DataFrame(
    [
        {
            'available': OPTUNA_NOTEBOOK_AVAILABLE,
            'detail': OPTUNA_NOTEBOOK_IMPORT_ERROR or 'Optuna wrappers imported successfully.',
            'imports': ', '.join(sorted(OPTUNA_IMPORTS)) if OPTUNA_IMPORTS else '',
        }
    ]
)

import_inventory = pd.DataFrame(
    [
        {'category': 'features', 'count': len(FEATURE_IMPORTS), 'imported_names': ', '.join(sorted(FEATURE_IMPORTS))},
        {'category': 'models', 'count': len(MODEL_IMPORTS), 'imported_names': ', '.join(sorted(MODEL_IMPORTS))},
        {'category': 'targets', 'count': len(TARGET_IMPORTS), 'imported_names': ', '.join(sorted(TARGET_IMPORTS))},
        {'category': 'signals', 'count': len(SIGNAL_IMPORTS), 'imported_names': ', '.join(sorted(SIGNAL_IMPORTS))},
        {'category': 'optuna_search', 'count': len(OPTUNA_IMPORTS), 'imported_names': ', '.join(sorted(OPTUNA_IMPORTS)) if OPTUNA_IMPORTS else OPTUNA_NOTEBOOK_IMPORT_ERROR},
    ]
)


## Catalogs And Imports

The next cell gives you two views:

- what is already imported into this notebook,
- what the experiment runner actually recognizes through its registries.


In [ ]:
show_df('Imported callables available in this notebook', import_inventory)
show_df('Tree-model runtime probes', tree_runtime_status)
show_df('Optuna notebook support', optuna_runtime_status)
show_df('Feature registry catalog', feature_catalog, rows=len(feature_catalog))
show_df('Model registry catalog', model_catalog, rows=len(model_catalog))
show_df('Target builder catalog', target_catalog, rows=len(target_catalog))
show_df('Signal registry catalog', signal_catalog, rows=len(signal_catalog))


## Raw Data

Edit only the path, symbol, or date range if you want a different sample. The examples below use the same repo-local CSV loading path as the existing feature lab.


In [ ]:
DATA_PATH = repo_root / 'data/raw/dukas_copy_bank/usdjpy_h1.csv'
DATA_SYMBOL = 'USDJPY'
START = '2017-05-07 23:00:00'
END = '2024-12-31 14:00:00'

raw_frame = load_ohlcv_csv(DATA_PATH, symbol=DATA_SYMBOL, start=START, end=END)

pd.DataFrame(
    {
        'rows': [len(raw_frame)],
        'columns': [len(raw_frame.columns)],
        'start': [raw_frame.index.min()],
        'end': [raw_frame.index.max()],
    }
)


## Feature Playground

Pick any `FEATURE_KIND` from the catalog and modify `FEATURE_PARAMS`. The helper shows you exactly which new columns the feature step created.


In [ ]:
FEATURE_KIND = 'trend'
FEATURE_PARAMS = {
    'price_col': 'close',
    'sma_windows': [20, 50],
    'ema_spans': [10, 20],
}

feature_fn = FEATURE_REGISTRY[FEATURE_KIND]
feature_frame, feature_new_cols = apply_feature(raw_frame, feature_fn, **FEATURE_PARAMS)

show_df(
    f'Feature catalog entry: {FEATURE_KIND}',
    feature_catalog.loc[feature_catalog['kind'] == FEATURE_KIND],
)
show_df(
    f'New columns from {FEATURE_KIND}',
    pd.Series(feature_new_cols, name='new_columns'),
    rows=max(len(feature_new_cols), 5),
)
feature_frame[existing_columns(feature_frame, ['open', 'high', 'low', 'close', 'volume'] + feature_new_cols)].tail(12)


## Target Playground

This cell lets you switch between `forward_return`, `triple_barrier`, and the classifier router. For `triple_barrier`, keep the anti-leakage assumptions explicit: price inputs are current-bar, labels depend only on future bars, and tail rows remain unlabeled when the full horizon is unavailable.


In [ ]:
TARGET_KIND = 'triple_barrier'  # forward_return | triple_barrier | classifier_router
TARGET_CFG = {
    'kind': TARGET_KIND,
    'price_col': 'close',
    'open_col': 'open',
    'high_col': 'high',
    'low_col': 'low',
    'returns_col': 'close_ret',
    'max_holding': 24,
    'upper_mult': 1.5,
    'lower_mult': 1.5,
    'vol_window': 24,
    'min_vol': 1e-4,
    'label_col': 'tb_label',
    'event_ret_col': 'tb_event_ret',
}

target_base = add_close_returns(raw_frame, log=False, col_name='close_ret')
target_builder = TARGET_BUILDERS[TARGET_KIND]
target_frame, target_label_col, target_fwd_col, target_meta = target_builder(target_base.copy(), TARGET_CFG)

display(Markdown(f'### Target builder: {TARGET_KIND}'))
display(pd.DataFrame([target_meta]).T.rename(columns={0: 'value'}))

target_columns = [
    'close',
    target_label_col,
    target_fwd_col,
    f'{target_label_col}_hit_step',
    f'{target_label_col}_upper_barrier',
    f'{target_label_col}_lower_barrier',
]

target_frame[existing_columns(target_frame, target_columns)].tail(12)


## Model Playground

The default example uses `logistic_regression_clf` because it is the cheapest stable baseline for fast walk-forward experimentation. Replace `MODEL_KIND` and `MODEL_CFG` only after the feature/target contract is already behaving as expected.


In [ ]:
MODEL_SAMPLE_ROWS = 3500
MODEL_KIND = 'logistic_regression_clf'

model_base = raw_frame.tail(MODEL_SAMPLE_ROWS).copy()
model_base = add_close_returns(model_base, log=False, col_name='close_ret')
model_base = add_lagged_features(model_base, cols=['close_ret'], lags=[1, 2, 3, 6, 12])
model_base = add_trend_features(model_base, price_col='close', sma_windows=[20, 50], ema_spans=[10, 20])
model_base = add_rsi_features(model_base, price_col='close', windows=[14])
model_base = add_volatility_features(
    model_base,
    returns_col='close_ret',
    rolling_windows=[24],
    ewma_spans=[24],
    annualization_factor=None,
)
model_base = add_atr_features(model_base, window=14, add_over_price=True)

MODEL_FEATURE_COLS = [
    'lag_close_ret_1',
    'lag_close_ret_2',
    'lag_close_ret_3',
    'lag_close_ret_6',
    'lag_close_ret_12',
    'close_over_sma_20',
    'close_over_sma_50',
    'close_over_ema_10',
    'close_over_ema_20',
    'close_rsi_14',
    'vol_rolling_24',
    'vol_ewma_24',
    'atr_14',
    'atr_over_price_14',
]

MODEL_CFG = {
    'params': {'max_iter': 400, 'solver': 'lbfgs'},
    'feature_cols': MODEL_FEATURE_COLS,
    'target': {'kind': 'forward_return', 'price_col': 'close', 'horizon': 1},
    'runtime': {'seed': 7, 'deterministic': True, 'threads': 1, 'repro_mode': 'strict'},
    'split': {
        'method': 'walk_forward',
        'train_size': 1500,
        'test_size': 250,
        'step_size': 250,
        'expanding': True,
    },
}

model_fn = SINGLE_ASSET_MODEL_REGISTRY[MODEL_KIND]
model_out, _, model_meta = model_fn(model_base, MODEL_CFG)

model_summary = pd.DataFrame(
    [
        {
            'model_kind': model_meta.get('model_kind'),
            'split_method': model_meta.get('split_method'),
            'n_folds': model_meta.get('n_folds'),
            'label_col': model_meta.get('label_col'),
            'oos_rows': int(model_out.get('pred_is_oos', pd.Series(False, index=model_out.index)).sum()),
            'evaluation_rows': (model_meta.get('oos_classification_summary') or {}).get('evaluation_rows'),
            'roc_auc': (model_meta.get('oos_classification_summary') or {}).get('roc_auc'),
            'accuracy': (model_meta.get('oos_classification_summary') or {}).get('accuracy'),
        }
    ]
)

show_df('Model summary', model_summary)
show_df('OOS classification summary', pd.DataFrame([model_meta.get('oos_classification_summary', {})]))
model_out[existing_columns(model_out, ['close', 'label', 'pred_prob', 'pred_is_oos'])].tail(20)


## Signal Playground

The default example converts `pred_prob` into positions through `probability_threshold`. This is the cleanest first pass for studying how model probability calibration maps into actual trading actions.


In [ ]:
SIGNAL_KIND = 'probability_threshold'
SIGNAL_PARAMS = {
    'prob_col': 'pred_prob',
    'signal_col': 'signal_prob_demo',
    'upper': 0.55,
    'lower': 0.45,
    'upper_exit': 0.52,
    'lower_exit': 0.48,
    'mode': 'long_short_hold',
}

signal_fn = SIGNAL_REGISTRY[SIGNAL_KIND]
signal_series = signal_fn(model_out.copy(), **SIGNAL_PARAMS)
signal_frame = model_out.copy()
signal_frame[signal_series.name] = signal_series

signal_distribution = (
    signal_frame.loc[signal_frame['pred_is_oos'].fillna(False), signal_series.name]
    .value_counts(dropna=False)
    .rename_axis('signal')
    .reset_index(name='count')
)

show_df('Signal catalog entry', signal_catalog.loc[signal_catalog['kind'] == SIGNAL_KIND])
show_df('OOS signal distribution', signal_distribution, rows=len(signal_distribution))
signal_frame[existing_columns(signal_frame, ['close', 'pred_prob', 'pred_is_oos', signal_series.name])].tail(20)


## Freeform Explorer

Use this cell when you want to inspect one catalog row quickly while changing only `SECTION` and `NAME`.


In [ ]:
SECTION = 'features'  # features | models | targets | signals
NAME = 'trend'

catalog_lookup = {
    'features': feature_catalog,
    'models': model_catalog,
    'targets': target_catalog,
    'signals': signal_catalog,
}

selected_catalog = catalog_lookup[SECTION]
selected_catalog.loc[selected_catalog['kind'] == NAME]


## Optuna Feature Tuning

Use this section when you want real Optuna search against a fixed experiment config. The intended workflow here is **feature-parameter tuning only** while keeping the downstream target, model, and signal fixed.

The first cell shows the `features[]` index map of the selected experiment config so you can target paths like `features.1.params.ema_spans.0` safely. The second cell gives you a guarded Optuna template that will only run when you explicitly set `RUN_OPTUNA = True`.


In [ ]:
OPTUNA_CONFIG_PATH = repo_root / 'config/experiments/btcusd_1h_shock_meta_logistic.yaml'

optuna_base_cfg = load_experiment_config(OPTUNA_CONFIG_PATH)
optuna_feature_rows = pd.DataFrame(
    [
        {
            'feature_idx': idx,
            'step': step.get('step'),
            'params': step.get('params', {}),
        }
        for idx, step in enumerate(optuna_base_cfg.get('features', []))
    ]
)

show_df('Optuna base config feature index map', optuna_feature_rows, rows=len(optuna_feature_rows))


In [ ]:
RUN_OPTUNA = False

if not OPTUNA_NOTEBOOK_AVAILABLE:
    display(Markdown('### Optuna is not available in this environment'))
    print(OPTUNA_NOTEBOOK_IMPORT_ERROR)
else:
    OPTUNA_SEARCH_SPACE = [
        SearchDimension(
            name='trend_ema_span',
            path='features.1.params.ema_spans.0',
            kind='int',
            low=8,
            high=64,
            step=4,
        ),
        SearchDimension(
            name='bollinger_window',
            path='features.3.params.window',
            kind='int',
            low=12,
            high=72,
            step=4,
        ),
        SearchDimension(
            name='atr_window',
            path='features.4.params.window',
            kind='int',
            low=8,
            high=48,
            step=4,
        ),
        SearchDimension(
            name='shock_ret_z_threshold',
            path='features.5.params.ret_z_threshold',
            kind='float',
            low=1.0,
            high=3.5,
            step=0.25,
        ),
    ]

    OPTUNA_OBJECTIVE = ObjectiveSpec(
        metric_path='evaluation.primary_summary.sharpe',
        direction='maximize',
        constraints=[
            ConstraintPenalty(
                metric_path='evaluation.primary_summary.max_drawdown',
                op='lt',
                threshold=-0.30,
                penalty=1.0,
            ),
            ConstraintPenalty(
                metric_path='derived.trade_count',
                op='lt',
                threshold=50,
                penalty=1.0,
            ),
        ],
        stability_weight=1.0,
        stability_metric_path='metrics.sharpe',
        stability_std_penalty=1.0,
    )

    show_df(
        'Optuna search space',
        pd.DataFrame([dim.__dict__ for dim in OPTUNA_SEARCH_SPACE]),
        rows=len(OPTUNA_SEARCH_SPACE),
    )
    show_df('Optuna objective', pd.DataFrame([OPTUNA_OBJECTIVE.__dict__]))

    if RUN_OPTUNA:
        study = optimize_experiment(
            OPTUNA_CONFIG_PATH,
            search_space=OPTUNA_SEARCH_SPACE,
            objective=OPTUNA_OBJECTIVE,
            n_trials=20,
            sampler='tpe',
            seed=7,
            logging_enabled=False,
        )

        best_trial_summary = pd.DataFrame(
            [
                {
                    'best_value': study.best_value,
                    'best_params': study.best_params,
                    'n_trials': len(study.trials),
                }
            ]
        )
        show_df('Optuna best trial', best_trial_summary)
    else:
        display(Markdown('### Optuna run is disabled by default'))
        print('Set RUN_OPTUNA = True only after you confirm the config path, search space, and local optuna installation.')


## Notes

- Feature builders and signals expose real Python signatures, so the catalog updates automatically when those callables change.
- Models and targets mostly accept config dictionaries, so the notebook shows the contract keys you tune in project configs rather than pretending the raw function signature is enough.
- If you experiment with `triple_barrier`, keep the search space small. Tuning barriers, holding horizon, and many feature windows at the same time is more likely to produce selection overfit than robust signal.
- Use the logistic baseline first, then tree models, then heavier sequence models. That order gives you the cleanest read on whether the alpha is in the data or only in model flexibility.
- The Optuna section is deliberately feature-only. Use it to tune feature parameters on a fixed downstream experiment, not to search features, target, model, and signals all at once.
